# Winning Combinations

**Why**: Test if top ideas have synergy or interfere when combined.

| combo | hypothesis |
|---|---|
| revise + JEPA | visual ceiling push |
| revise + sparsity | sort ceiling via 2 mechanisms |
| JEPA + sparsity | representation synergy |
| full stack | all 3 combined |

**Hardware**: 1 machine x 8 GPUs (~8h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (single ideas overview)

How each idea performed individually, for comparison with combos below.

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    ideas = [('revise','st10','revise'), ('JEPA','st04','jepa_w0.1'),
             ('sparsity','st08','sparsity0.5')]
    for name, stage, sweep in ideas:
        for task in ['cifar10','sort','mazes']:
            sub = df_prior[(df_prior.stage==stage)&(df_prior.sweep==sweep)&(df_prior.task==task)]
            if not sub.empty:
                m = sub.best_test_acc.mean() * 100
                bl = BASELINE_ACC[task] * 100
                print(f'{name:10s} {task:10s}: {m:.1f}% ({m-bl:+.1f}pp)')
else:
    print('Prior data not found.')

## Part B - Combo Design (4 combos x 2 tasks x 3 seeds = 24 runs)

In [ ]:
COMBOS = [
    ('revise+jepa',     make_combo(['cifar10','mazes'], [0,1,2], use_revise=True, use_jepa=True)),
    ('revise+sparsity', make_combo(['sort','mazes'],    [0,1,2], use_revise=True, use_sparsity=True)),
    ('jepa+sparsity',   make_combo(['cifar10','sort'],  [0,1,2], use_jepa=True, use_sparsity=True)),
    ('full_stack',      make_combo(['cifar10','sort'],  [0,1,2], use_revise=True, use_jepa=True, use_sparsity=True)),
]
exps = []
for name, group in COMBOS:
    print(f'{name:20s}: {len(group)} runs')
    exps.extend(group)
print(f'\nTotal: {len(exps)}')

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/04_combos', dry_run=True)

## Part C - Run Training

Set `CONFIRM_RUN = True` to launch (~8h).

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/04_combos')


In [ ]:
status('logs/deep/04_combos')

## Part D - Results Analysis

In [ ]:
df = collect('logs/deep/04_combos')
if df.empty:
    print('No results yet.')
else:
    df['combo'] = df['name'].apply(lambda n: '+'.join([p for p in ['revise','jepa','spar'] if p in n]))
    print(df[['name','task','combo','best_acc','delta']].to_string(index=False))
    plot_delta_bars(df, 'Combos vs baseline', 'figures/04_delta.png')

In [ ]:
if not df.empty:
    import numpy as np
    agg = df.dropna(subset=['best_acc']).groupby(['combo','task'])['best_acc'].agg(['mean','std']).reset_index()
    combos_order = sorted(agg.combo.unique())
    tasks_order = sorted(agg.task.unique())
    x = np.arange(len(combos_order))
    w = 0.8 / max(len(tasks_order), 1)
    fig, ax = plt.subplots(figsize=(12, 5.5))
    for i, task in enumerate(tasks_order):
        vals = []
        for c in combos_order:
            row = agg[(agg.combo==c)&(agg.task==task)]
            vals.append(row['mean'].values[0]*100 if not row.empty else 0)
        ax.bar(x + i*w - 0.4 + w/2, vals, w, label=task, edgecolor='black', lw=0.4)
    ax.set_xticks(x); ax.set_xticklabels(combos_order, rotation=15)
    ax.set_ylabel('best test acc (%)'); ax.legend()
    ax.set_title('Combos grouped by task'); ax.grid(True, axis='y', alpha=0.2)
    fig.tight_layout(); fig.savefig('figures/04_grouped.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
if not df.empty:
    print(summary_stats(df, groupby=('combo','task')))